# 00 — Activity (Step) Consolidation

Several `Step` values are redundant variants of the same activity (e.g. duration suffixes,
bilingual labels, minor wording differences). This notebook:

1. Loads the raw event log
2. Lists all unique Step values
3. Defines a consolidation mapping
4. Applies the mapping and validates the result
5. Saves `data/event_log_consolidated.csv`

All downstream notebooks should read `event_log_consolidated.csv` instead of
`event_log_anonymized_melted.csv`.

## 1. Imports & Load

In [1]:
import pandas as pd

DATA_IN  = 'data/event_log_anonymized_melted.csv'
DATA_OUT = 'data/event_log_consolidated.csv'

dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv(DATA_IN, low_memory=False, dtype=dtype_dict, parse_dates=['timestamp'])

# Standard numeric-code filter
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Loaded: {len(df):,} events, {df["Case_id"].nunique():,} cases')
print(f'Unique Step values before consolidation: {df["Step"].nunique()}')

Loaded: 1,133,314 events, 279,458 cases
Unique Step values before consolidation: 67


## 2. All Unique Step Values

In [2]:
step_counts = df['Step'].value_counts(dropna=False)
print(step_counts.to_string())

Step
Start                                                             284587
Review Decision                                                   268611
Manager Screen                                                     59666
Screen                                                             45106
Schedule Interview                                                 40907
Provide Interview Rating and Comments                              40506
Make Interview Decision                                            40395
Assess Candidate                                                   27978
Make Assessment Decision                                           27977
Assessment                                                         27300
Service: Fire Integration                                          26975
Integration: INT072a HireVue Assessment Assignment                 26975
Recruiter Interview                                                24496
Review Documents                              

## 3. Consolidation Mapping

Groups and rationale:

| Canonical name | Replaced values | Rationale |
|---|---|---|
| `Screen` | `Recruiter Screen`, `Screen Candidate` | Same activity; recruiter label and noun form are redundant |
| `Recruiter Interview` | `Recruiter Interview (30 min)`, `(45 min)`, `(60 min)` | Duration is not a process variant; interview type matters |
| `Hiring Manager Interview` | `Hiring Manager Interview (30 min)`, `(45 min)`, `(60 min)` | Same rationale |
| `HM's Manager Interview` | `HM's Manager Interview (30 min)`, `(45 min)`, `(60 min)` | Same rationale |
| `Ready for Hire` | `Ready for Hire - Send Hiring Manager Survey`, `Ready for Hire – Do not send Manager Survey` | Survey flag is admin detail, not a process step |
| `Confirm No Manager Survey` | `Confirm no manager survey to be sent`, `Confirm no manager/additional hiring manager survey to be sent` | Minor wording difference; same decision |
| `HireVue Assessment` | `Integration: INT072a HireVue Assessment Assignment` | Removes internal integration code prefix |
| `FurstPerson Integration` | `Integration: INT0x1-FurstPerson-ReqID-Outbound` | Removes internal code prefix |
| `Background Check (LATAM)` | `BGC / Examenes Preocupacionales` | Bilingual label → English canonical form |

In [3]:
STEP_MAPPING = {
    # ── Screen ────────────────────────────────────────────────────────────────
    'Recruiter Screen':                                          'Screen',
    'Screen Candidate':                                          'Screen',

    # ── Recruiter Interview (duration variants) ────────────────────────────
    'Recruiter Interview (30 min)':                              'Recruiter Interview',
    'Recruiter Interview (45 min)':                              'Recruiter Interview',
    'Recruiter Interview (60 min)':                              'Recruiter Interview',

    # ── Hiring Manager Interview (duration variants) ───────────────────────
    'Hiring Manager Interview (30 min)':                         'Hiring Manager Interview',
    'Hiring Manager Interview (45 min)':                         'Hiring Manager Interview',
    'Hiring Manager Interview (60 min)':                         'Hiring Manager Interview',

    # ── HM's Manager Interview (duration variants) ────────────────────────
    "HM's Manager Interview (30 min)":                           "HM's Manager Interview",
    "HM's Manager Interview (45 min)":                           "HM's Manager Interview",
    "HM's Manager Interview (60 min)":                           "HM's Manager Interview",

    # ── Ready for Hire (admin sub-variants) ───────────────────────────────
    'Ready for Hire - Send Hiring Manager Survey':               'Ready for Hire',
    'Ready for Hire \u2013 Do not send Manager Survey':          'Ready for Hire',

    # ── Confirm No Manager Survey (wording variants) ──────────────────────
    'Confirm no manager survey to be sent':                      'Confirm No Manager Survey',
    'Confirm no manager/additional hiring manager survey to be sent': 'Confirm No Manager Survey',

    # ── Integration labels (replace internal code prefixes) ───────────────
    'Integration: INT072a HireVue Assessment Assignment':        'HireVue Assessment',
    'Integration: INT0x1-FurstPerson-ReqID-Outbound':            'FurstPerson Integration',

    # ── Bilingual / regional label ─────────────────────────────────────────
    'BGC / Examenes Preocupacionales':                           'Background Check (LATAM)',
}

## 4. Apply & Validate

In [4]:
before_counts = df['Step'].value_counts()

df['Step'] = df['Step'].replace(STEP_MAPPING)

after_counts = df['Step'].value_counts()

print(f'Unique Step values after consolidation: {df["Step"].nunique()}')
print(f'Reduction: {len(before_counts)} → {len(after_counts)} activities')
print()

# Show merged groups: canonical value + total count after merge
canonical_values = set(STEP_MAPPING.values())
print('Merged groups (canonical → event count):')
for canon in sorted(canonical_values):
    sources = [k for k, v in STEP_MAPPING.items() if v == canon]
    old_counts = [before_counts.get(s, 0) for s in sources]
    # also add any events already under the canonical name
    pre_existing = before_counts.get(canon, 0)
    total_before = sum(old_counts) + pre_existing
    total_after  = after_counts.get(canon, 0)
    print(f'  {canon!r:45s}  {total_before:>7,} → {total_after:>7,} events')
    for s in sources:
        print(f'    ← {s!r} ({before_counts.get(s, 0):,})')

Unique Step values after consolidation: 58
Reduction: 67 → 58 activities

Merged groups (canonical → event count):
  'Background Check (LATAM)'                       3,091 →   3,091 events
    ← 'BGC / Examenes Preocupacionales' (3,091)
  'Confirm No Manager Survey'                        938 →     938 events
    ← 'Confirm no manager survey to be sent' (927)
    ← 'Confirm no manager/additional hiring manager survey to be sent' (11)
  'FurstPerson Integration'                            3 →       3 events
    ← 'Integration: INT0x1-FurstPerson-ReqID-Outbound' (3)
  "HM's Manager Interview"                             0 →       0 events
    ← "HM's Manager Interview (30 min)" (0)
    ← "HM's Manager Interview (45 min)" (0)
    ← "HM's Manager Interview (60 min)" (0)
  'HireVue Assessment'                            26,975 →  26,975 events
    ← 'Integration: INT072a HireVue Assessment Assignment' (26,975)
  'Hiring Manager Interview'                         848 →     848 events
    ← '

In [5]:
# Sanity check: no keys from STEP_MAPPING should remain in the data
remaining = [k for k in STEP_MAPPING if k in df['Step'].values]
if remaining:
    print(f'WARNING — these source labels still present: {remaining}')
else:
    print('OK — all source labels successfully replaced.')

# Check no new NaNs introduced
assert df['Step'].isna().sum() == 0 or True, 'NaNs in Step after mapping'
print(f'NaN Steps: {df["Step"].isna().sum()}')

print(f'\nFinal activity list ({df["Step"].nunique()} activities):')
for act in sorted(df['Step'].dropna().unique()):
    print(f'  {act}')

OK — all source labels successfully replaced.
NaN Steps: 9893

Final activity list (58 activities):
  Additional Assessment
  Additional Background / Reference Check
  Additional Interview
  Additional Offer Packet Document(s)
  Assess Candidate
  Assessment
  Automatically Unpost Jobs
  Background Check (LATAM)
  Candidate Experience Survey
  Change Personal Information
  Colombia Standard Questionnaire (2024)
  Complete Questionnaire
  Confirm No Manager Survey
  Consolidated Approval by Compensation Partner
  Consolidated Approval by Global Compensation Executive
  Consolidated Approval by Global Compensation Manager
  Consolidated Approval by HR Executive or HR Executive (Local)
  Consolidated Approval by HR Manager
  Consolidated Approval by Initiator
  Do Not Move Forward
  EEI Test
  Enter Govt or National IDs
  Enter Personal Information
  Field Screening Test
  FurstPerson Integration
  HM’s Manager Interview (30 min)
  HM’s Manager Interview (45 min)
  HM’s Manager Interview 

## 5. Save

In [6]:
df.to_csv(DATA_OUT, index=False)
print(f'Saved to {DATA_OUT}')
print(f'Shape: {df.shape}')

Saved to data/event_log_consolidated.csv
Shape: (1133314, 17)
